# Análisis 1 — Tendencia, estacionalidad y quiebre estructural

**Objetivo:** Identificar el punto de quiebre en las llegadas internacionales a Ecuador y cuantificar cómo la crisis de seguridad 2022-2024 alteró la tendencia y el patrón estacional del turismo receptor.

**Fuentes:**
- Banco Mundial ST.INT.ARVL — llegadas anuales Ecuador 2010-2019
- ESI INEC — registros individuales 2023-2025 (extranjeros, motivo turismo)
- MDI — homicidios intencionales 2014-2025 (registros individuales por fecha y provincia)
- UNODC/BM — tasa de homicidios por 100.000 hab. (serie histórica)

**Nota sobre el gap 2020-2022:** El Banco Mundial no reporta llegadas para Ecuador en 2020-2025. La ESI comienza en 2023. Los años 2020-2022 (COVID + inicio de crisis de seguridad) no tienen datos de llegadas disponibles en fuentes abiertas con desagregación suficiente. Se documenta esta limitación explícitamente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import duckdb
import zipfile
from statsmodels.tsa.seasonal import STL
from sklearn.metrics import mean_absolute_percentage_error

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print('Librerías cargadas')

## 1. Cargar datos de homicidios (MDI 2014-2025)

In [ ]:
# Datos individuales MDI: registros por incidente con fecha y provincia
df_mdi = pd.read_excel(
    '../data/raw/seguridad/mdi_homicidiosintencionales_pm_2014_2025.xlsx',
    sheet_name='1. Homicidios Intencionales'
)

print(f'Total registros MDI: {len(df_mdi):,}')
print(f'Columnas clave: fecha_infraccion, provincia, tipo_muerte')

# Parsear fecha
df_mdi['fecha'] = pd.to_datetime(df_mdi['fecha_infraccion'], errors='coerce')
df_mdi['anio'] = df_mdi['fecha'].dt.year
df_mdi['mes'] = df_mdi['fecha'].dt.month

# Verificar rango temporal
print(f'\nRango temporal: {df_mdi["fecha"].min().date()} → {df_mdi["fecha"].max().date()}')
print(f'Registros con fecha nula: {df_mdi["fecha"].isna().sum()}')

In [ ]:
# Serie anual de homicidios (conteo total)
hom_anual = (
    df_mdi.dropna(subset=['anio'])
    .groupby('anio')
    .size()
    .reset_index(name='homicidios')
)
hom_anual = hom_anual[hom_anual['anio'].between(2014, 2025)]

# Serie mensual de homicidios (para overlay con ESI)
hom_mensual = (
    df_mdi.dropna(subset=['anio', 'mes'])
    .groupby(['anio', 'mes'])
    .size()
    .reset_index(name='homicidios')
)
hom_mensual['fecha'] = pd.to_datetime(
    hom_mensual['anio'].astype(str) + '-' + hom_mensual['mes'].astype(str).str.zfill(2) + '-01'
)
hom_mensual = hom_mensual[hom_mensual['anio'].between(2014, 2025)].sort_values('fecha')

print('Serie anual homicidios:')
print(hom_anual.to_string(index=False))

In [ ]:
# Cargar también la tasa por 100.000 hab. (UNODC/Banco Mundial)
with zipfile.ZipFile('../data/raw/seguridad/API_VC.IHR.PSRC.P5_DS2_es_csv_v2_8348.zip') as z:
    csv_name = [n for n in z.namelist() if n.endswith('.csv') and 'Metadata' not in n][0]
    with z.open(csv_name) as f:
        raw_unodc = pd.read_csv(f, skiprows=4, encoding='utf-8')

ecu_hom_tasa = raw_unodc[raw_unodc['Country Code'] == 'ECU'].copy()
years_unodc = [str(y) for y in range(2010, 2025)]
ecu_hom_tasa_long = (
    ecu_hom_tasa[['Country Name'] + years_unodc]
    .melt(id_vars='Country Name', var_name='anio', value_name='tasa_hom_100k')
    .dropna(subset=['tasa_hom_100k'])
)
ecu_hom_tasa_long['anio'] = ecu_hom_tasa_long['anio'].astype(int)

print('Tasa de homicidios por 100.000 hab. — Ecuador:')
print(ecu_hom_tasa_long[['anio', 'tasa_hom_100k']].to_string(index=False))

## 2. Cargar llegadas internacionales (Banco Mundial 2010-2019)

In [ ]:
with zipfile.ZipFile('../data/raw/banco_mundial/API_ST.INT.ARVL_DS2_es_csv_v2_4967.zip') as z:
    fname = [n for n in z.namelist() if n.endswith('.csv') and 'Metadata' not in n][0]
    with z.open(fname) as f:
        raw_bm = pd.read_csv(f, skiprows=4, encoding='utf-8')

ecu_bm = raw_bm[raw_bm['Country Code'] == 'ECU'].copy()
bm_years = [str(y) for y in range(2010, 2020)]  # Solo hasta 2019 (2020+ son NaN)

bm_long = (
    ecu_bm[['Country Name'] + bm_years]
    .melt(id_vars='Country Name', var_name='anio', value_name='llegadas')
    .dropna(subset=['llegadas'])
)
bm_long['anio'] = bm_long['anio'].astype(int)
bm_long['llegadas'] = bm_long['llegadas'].astype(int)

print('Llegadas internacionales Ecuador — Banco Mundial:')
print(bm_long[['anio', 'llegadas']].to_string(index=False))
print(f'\nNota: datos disponibles 2010-2019 únicamente (2020+ = NaN en fuente BM)')

## 3. Cargar llegadas ESI 2023-2025 (extranjeros, motivo turismo)

In [ ]:
con = duckdb.connect()

ESI_2023 = "../data/raw/entradas_salidas/Datos_abiertos_ ESI_2023/esi2023.csv"
ESI_2024 = "../data/raw/entradas_salidas/Datos_abiertos_ ESI_2024/esi_2024.csv"
ESI_2025 = "../data/raw/entradas_salidas/Datos_abiertos_ESI_2025/ESI2025.csv"

# Llegadas de extranjeros por año y mes
# Filtro amplio: Entrada + Extranjero (incluye todos los motivos)
# También extraemos el subconjunto de mot_viam='Turismo' para análisis más preciso
df_esi_mensual = con.execute(f"""
    SELECT
        anio_movi AS anio,
        mes_movi AS mes_nombre,
        mot_viam AS motivo,
        COUNT(*) AS total
    FROM (
        SELECT anio_movi, mes_movi, mot_viam, tip_movi, tip_naci FROM read_csv_auto('{ESI_2023}')
        UNION ALL
        SELECT anio_movi, mes_movi, mot_viam, tip_movi, tip_naci FROM read_csv_auto('{ESI_2024}')
        UNION ALL
        SELECT anio_movi, mes_movi, mot_viam, tip_movi, tip_naci FROM read_csv_auto('{ESI_2025}')
    )
    WHERE tip_movi = 'Entrada' AND tip_naci = 'Extranjero'
    GROUP BY anio_movi, mes_movi, mot_viam
    ORDER BY anio_movi, mes_movi
""").df()

# Mapeo mes nombre → número
meses_map = {
    'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4,
    'Mayo': 5, 'Junio': 6, 'Julio': 7, 'Agosto': 8,
    'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12
}
df_esi_mensual['mes'] = df_esi_mensual['mes_nombre'].map(meses_map)
df_esi_mensual['fecha'] = pd.to_datetime(
    df_esi_mensual['anio'].astype(str) + '-' +
    df_esi_mensual['mes'].astype(str).str.zfill(2) + '-01'
)

# Total extranjeros por mes (todos los motivos)
esi_total_mes = (
    df_esi_mensual.groupby(['fecha', 'anio', 'mes'])
    ['total'].sum().reset_index()
    .sort_values('fecha')
)

# Solo turismo
esi_turismo_mes = (
    df_esi_mensual[df_esi_mensual['motivo'].str.contains('Turismo', na=False)]
    .groupby(['fecha', 'anio', 'mes'])
    ['total'].sum().reset_index()
    .sort_values('fecha')
)

# Totales anuales ESI
esi_anual = esi_total_mes.groupby('anio')['total'].sum().reset_index()
esi_turismo_anual = esi_turismo_mes.groupby('anio')['total'].sum().reset_index()

print('Llegadas ESI (todos los motivos) por año:')
print(esi_anual.to_string(index=False))
print()
print('Llegadas ESI (solo turismo) por año:')
print(esi_turismo_anual.to_string(index=False))

# Distribución de motivos
print()
print('Distribución de motivos de viaje (extranjeros, 2023-2025):')
motivos_dist = df_esi_mensual.groupby('motivo')['total'].sum().sort_values(ascending=False)
motivos_dist_pct = (motivos_dist / motivos_dist.sum() * 100).round(1)
print(pd.DataFrame({'conteo': motivos_dist, 'pct': motivos_dist_pct}).to_string())

## 4. Serie de tendencia larga: llegadas Ecuador 2010-2025

Combinamos Banco Mundial (2010-2019) con ESI (2023-2025).
El período 2020-2022 se muestra como zona sombreada (datos no disponibles en fuentes abiertas).

In [ ]:
# Construir serie larga combinada
# BM: llegadas TOTALES de turistas internacionales (definición UNWTO)
# ESI: llegadas de extranjeros (incluye todos los motivos)
# Nota: las cifras no son directamente comparables, se muestran en ejes normalizados

serie_bm = bm_long[['anio', 'llegadas']].copy()
serie_bm['fuente'] = 'Banco Mundial'

serie_esi = esi_anual.rename(columns={'total': 'llegadas'}).copy()
serie_esi['fuente'] = 'ESI/INEC'

serie_larga = pd.concat([serie_bm, serie_esi], ignore_index=True).sort_values('anio')

# Gráfico de tendencia larga
fig, ax1 = plt.subplots(figsize=(14, 6))

# --- Zonas sombreadas ---
ax1.axvspan(2019.5, 2022.5, alpha=0.12, color='gray', label='Sin datos llegadas (2020-2022)')
ax1.axvspan(2021.5, 2024.5, alpha=0.10, color='red', label='Crisis de seguridad (2022-2024)')
ax1.axvspan(2019.8, 2021.2, alpha=0.15, color='blue', label='Pandemia COVID-19 (2020-2021)')

# --- Línea BM ---
bm_plot = serie_larga[serie_larga['fuente'] == 'Banco Mundial']
ax1.plot(bm_plot['anio'], bm_plot['llegadas'], 'o-',
         color='steelblue', linewidth=2.5, markersize=7, label='Llegadas BM (turistas intl.)')

# --- Línea ESI ---
esi_plot = serie_larga[serie_larga['fuente'] == 'ESI/INEC']
ax1.plot(esi_plot['anio'], esi_plot['llegadas'], 's--',
         color='darkorange', linewidth=2.5, markersize=7, label='Entradas ESI (extranjeros)')

# Anotar punto máximo pre-crisis
max_bm = bm_plot.loc[bm_plot['llegadas'].idxmax()]
ax1.annotate(f"Pico 2018\n{max_bm['llegadas']/1e6:.2f}M",
             xy=(max_bm['anio'], max_bm['llegadas']),
             xytext=(max_bm['anio']-1.5, max_bm['llegadas']+80000),
             fontsize=9, color='steelblue',
             arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.5))

ax1.set_xlabel('Año', fontsize=12)
ax1.set_ylabel('Llegadas internacionales', fontsize=12)
ax1.set_title('Ecuador: Llegadas de turistas internacionales 2010-2025\n'
              '(Banco Mundial 2010-2019 | ESI/INEC 2023-2025)', fontsize=13)
ax1.set_xticks(range(2010, 2026))
ax1.tick_params(axis='x', rotation=45)
ax1.legend(loc='upper left', fontsize=9)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# --- Eje secundario: tasa de homicidios ---
ax2 = ax1.twinx()
ax2.plot(ecu_hom_tasa_long['anio'], ecu_hom_tasa_long['tasa_hom_100k'],
         '^-', color='crimson', linewidth=2, markersize=6, label='Homicidios / 100k hab. (UNODC)')
ax2.set_ylabel('Tasa de homicidios por 100.000 hab.', fontsize=12, color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/01_tendencia_larga_llegadas_homicidios.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en outputs/')

## 5. Evolución anual de homicidios (serie MDI 2014-2025)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=False)

# --- Anual ---
colores_anual = ['#d62728' if a >= 2022 else '#1f77b4' for a in hom_anual['anio']]
bars = ax1.bar(hom_anual['anio'], hom_anual['homicidios'], color=colores_anual, alpha=0.85)
for bar, val in zip(bars, hom_anual['homicidios']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}', ha='center', va='bottom', fontsize=8)
ax1.set_title('Homicidios intencionales en Ecuador — Serie anual (MDI 2014-2025)', fontsize=12)
ax1.set_ylabel('Número de homicidios')
ax1.set_xticks(hom_anual['anio'])
ax1.tick_params(axis='x', rotation=45)
patch1 = mpatches.Patch(color='#1f77b4', label='Pre-crisis (2014-2021)')
patch2 = mpatches.Patch(color='#d62728', label='Crisis de seguridad (2022+)')
ax1.legend(handles=[patch1, patch2], fontsize=9)
ax1.axvline(x=2021.5, color='black', linestyle='--', linewidth=1.5, alpha=0.6)

# --- Mensual (2020-2025) ---
hom_mens_reciente = hom_mensual[hom_mensual['anio'] >= 2019].sort_values('fecha')
ax2.plot(hom_mens_reciente['fecha'], hom_mens_reciente['homicidios'],
         color='crimson', linewidth=1.5, alpha=0.8)
ax2.fill_between(hom_mens_reciente['fecha'], hom_mens_reciente['homicidios'],
                 alpha=0.2, color='crimson')
ax2.axvline(pd.Timestamp('2022-01-01'), color='black', linestyle='--',
            linewidth=1.5, label='Inicio escalada (ene 2022)')
ax2.set_title('Serie mensual de homicidios (MDI 2019-2025)', fontsize=12)
ax2.set_ylabel('Homicidios por mes')
ax2.legend(fontsize=9)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/02_serie_homicidios_anual_mensual.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. STL Decomposition — Llegadas mensuales de extranjeros (ESI 2023-2025)

Con 36 meses de datos aplicamos STL para aislar tendencia, estacionalidad y ruido.
Se usa period=12 (ciclo anual). Los datos 2023-2025 ya están dentro de la crisis, por lo que la componente de tendencia refleja la dinámica de recuperación parcial.

In [ ]:
# Serie mensual completa ordenada
ts_total = esi_total_mes.set_index('fecha')['total'].sort_index()
ts_turismo = esi_turismo_mes.set_index('fecha')['total'].sort_index()

print(f'Serie total: {len(ts_total)} meses ({ts_total.index.min().date()} → {ts_total.index.max().date()})')
print(f'Serie turismo: {len(ts_turismo)} meses')
print(f'\nPrimeros valores:')
print(ts_total.head(6))

# Verificar valores nulos
print(f'\nValores nulos en serie total: {ts_total.isna().sum()}')
print(f'Valores nulos en serie turismo: {ts_turismo.isna().sum()}')

In [ ]:
# STL sobre serie total (extranjeros, todos los motivos)
stl_total = STL(ts_total, period=12, robust=True)
result_total = stl_total.fit()

# STL sobre serie turismo
stl_turismo = STL(ts_turismo, period=12, robust=True)
result_turismo = stl_turismo.fit()

# Gráfico STL — serie total
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

axes[0].plot(ts_total.index, ts_total.values, color='steelblue', linewidth=1.5)
axes[0].set_title('Serie observada — Entradas de extranjeros (todos los motivos)', fontsize=10)
axes[0].set_ylabel('Entradas')

axes[1].plot(ts_total.index, result_total.trend, color='darkorange', linewidth=2)
axes[1].set_title('Componente de tendencia (STL)', fontsize=10)
axes[1].set_ylabel('Tendencia')

axes[2].plot(ts_total.index, result_total.seasonal, color='green', linewidth=1.5)
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_title('Componente estacional (STL)', fontsize=10)
axes[2].set_ylabel('Estacionalidad')

axes[3].plot(ts_total.index, result_total.resid, color='gray', linewidth=1)
axes[3].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[3].set_title('Componente residual (ruido)', fontsize=10)
axes[3].set_ylabel('Residuo')
axes[3].tick_params(axis='x', rotation=45)

fig.suptitle('Descomposición STL — Entradas internacionales Ecuador (2023-2025)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/03_stl_decomposition_total.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Métrica MAPE: reconstrucción de la serie a partir de las componentes
ts_reconstruida = result_total.trend + result_total.seasonal + result_total.resid
mape_total = mean_absolute_percentage_error(ts_total.values, ts_reconstruida.values) * 100
print(f'MAPE (reconstrucción STL serie total): {mape_total:.4f}%')
print(f'→ Resultado: {"PASA" if mape_total < 15 else "NO PASA"} (umbral < 15%)')

# STL turismo
ts_rec_turismo = result_turismo.trend + result_turismo.seasonal + result_turismo.resid
mape_turismo = mean_absolute_percentage_error(ts_turismo.values, ts_rec_turismo.values) * 100
print(f'\nMAPE (reconstrucción STL serie turismo): {mape_turismo:.4f}%')
print(f'→ Resultado: {"PASA" if mape_turismo < 15 else "NO PASA"} (umbral < 15%)')

## 7. Índice de estacionalidad mensual

In [ ]:
MESES_NOMBRE = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

# Índice estacional por mes (promedio de la componente estacional STL)
df_seasonal = pd.DataFrame({
    'fecha': ts_total.index,
    'mes': ts_total.index.month,
    'estacional': result_total.seasonal
})

indice_estacional = df_seasonal.groupby('mes')['estacional'].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Barras de índice estacional
colores_bar = ['#d62728' if v > 0 else '#1f77b4' for v in indice_estacional.values]
ax1.bar(range(1, 13), indice_estacional.values, color=colores_bar, alpha=0.85)
ax1.axhline(0, color='black', linewidth=1)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(MESES_NOMBRE)
ax1.set_title('Índice estacional mensual\n(Componente STL — Entradas extranjeros 2023-2025)', fontsize=10)
ax1.set_ylabel('Desviación estacional respecto a la tendencia')

# Comparativa por año y mes
esi_pivot = esi_total_mes.copy()
esi_pivot['mes_nombre'] = esi_pivot['mes'].map({i+1: m for i, m in enumerate(MESES_NOMBRE)})
pivot = esi_pivot.pivot_table(index='mes', columns='anio', values='total').fillna(0)

for anio in pivot.columns:
    ax2.plot(pivot.index, pivot[anio], marker='o', label=str(anio), linewidth=1.8)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(MESES_NOMBRE)
ax2.set_title('Entradas de extranjeros por mes y año\n(ESI 2023-2025)', fontsize=10)
ax2.set_ylabel('Número de entradas')
ax2.legend(title='Año')

plt.tight_layout()
plt.savefig('../outputs/04_indice_estacional.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nÍndice estacional por mes (promedio):')
for mes, val in indice_estacional.items():
    signo = '↑ ALTA TEMPORADA' if val > 2000 else ('↓ BAJA TEMPORADA' if val < -2000 else '  Temporada media')
    print(f'  {MESES_NOMBRE[mes-1]:>3}: {val:>+10,.0f}  {signo}')

## 8. Overlay: llegadas mensuales ESI + homicidios mensuales (2023-2025)

Visualización de la evolución conjunta en el único período con datos individuales en ambas series.

In [ ]:
# Cruzar homicidios mensuales con llegadas mensuales (2023-2025)
hom_mens_2325 = hom_mensual[hom_mensual['anio'].between(2023, 2025)].copy()

df_overlay = esi_total_mes.merge(
    hom_mens_2325[['fecha', 'homicidios']],
    on='fecha', how='inner'
).sort_values('fecha')

print(f'Meses en common: {len(df_overlay)}')

fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

ax1.plot(df_overlay['fecha'], df_overlay['total'],
         'o-', color='steelblue', linewidth=2, markersize=5, label='Entradas extranjeros (ESI)')
ax1.fill_between(df_overlay['fecha'], df_overlay['total'], alpha=0.1, color='steelblue')

ax2.plot(df_overlay['fecha'], df_overlay['homicidios'],
         's--', color='crimson', linewidth=1.8, markersize=5, label='Homicidios mensuales (MDI)')

ax1.set_xlabel('Mes/Año')
ax1.set_ylabel('Entradas de extranjeros', color='steelblue')
ax2.set_ylabel('Homicidios por mes', color='crimson')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='crimson')
ax1.tick_params(axis='x', rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax1.set_title('Entradas internacionales vs Homicidios mensuales — Ecuador 2023-2025', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/05_overlay_entradas_homicidios_mensual.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlación
corr_mensual = df_overlay['total'].corr(df_overlay['homicidios'])
print(f'\nCorrelación de Pearson mensual (llegadas vs homicidios 2023-2025): {corr_mensual:.3f}')

## 9. Resumen ejecutivo del análisis

In [ ]:
print('=' * 60)
print('RESUMEN — Análisis 1: Tendencia y Estacionalidad')
print('=' * 60)

print('\n📊 DATOS UTILIZADOS:')
print(f'  • Banco Mundial (llegadas anuales): 2010-2019')
print(f'  • ESI/INEC (llegadas mensuales):    2023-2025')
print(f'  • MDI (homicidios mensuales):       2014-2025')
print(f'  • UNODC (tasa homicidios/100k):     2010-{ecu_hom_tasa_long["anio"].max()}')

print('\n📉 TENDENCIA:')
pico_bm = bm_long.loc[bm_long['llegadas'].idxmax()]
print(f'  • Pico pre-crisis: {pico_bm["llegadas"]:,} llegadas ({int(pico_bm["anio"])})')
print(f'  • Entradas ESI 2023: {esi_anual[esi_anual["anio"]==2023]["total"].values[0]:,} (extranjeros, todos motivos)')
print(f'  • Limitación: gap de datos 2020-2022 (no disponibles en fuentes abiertas)')

print('\n🌡️ ESTACIONALIDAD (STL):')
mes_pico = indice_estacional.idxmax()
mes_valle = indice_estacional.idxmin()
print(f'  • Mes de mayor afluencia:   {MESES_NOMBRE[mes_pico-1]} (índice: +{indice_estacional[mes_pico]:,.0f})')
print(f'  • Mes de menor afluencia:   {MESES_NOMBRE[mes_valle-1]} (índice: {indice_estacional[mes_valle]:,.0f})')
print(f'  • MAPE reconstrucción STL:  {mape_total:.4f}% → {"PASA" if mape_total < 15 else "NO PASA"} (umbral 15%)')

print('\n🔴 HOMICIDIOS:')
hom_pico = hom_anual.loc[hom_anual['homicidios'].idxmax()]
hom_base = hom_anual[hom_anual['anio'] == 2019]['homicidios'].values[0]
print(f'  • Línea base 2019:    {hom_base:,} homicidios')
print(f'  • Pico crisis:        {int(hom_pico["homicidios"]):,} homicidios ({int(hom_pico["anio"])})')
print(f'  • Multiplicador:      {hom_pico["homicidios"]/hom_base:.1f}x respecto a 2019')
print(f'  • Correlación mensual (2023-2025): {corr_mensual:.3f}')

print('\n✅ MÉTRICAS DE ÉXITO:')
print(f'  • MAPE < 15%:     {"✓" if mape_total < 15 else "✗"} ({mape_total:.4f}%)')

print('\n⚠️ LIMITACIONES IDENTIFICADAS:')
print('  • El gap 2020-2022 impide cuantificar el impacto COVID + inicio crisis')
print('  • Los datos BM y ESI no son directamente comparables (definiciones distintas)')
print('  • Con 36 meses de ESI, el STL opera en el límite mínimo recomendado')
print('  • La correlación mensual 2023-2025 solo captura el período de crisis, no la caída')